# Online Retail Sales Performance Analysis

**Business question:** How is revenue trending over time, which products and regions drive the most value, and where are the opportunities to grow average order value?

**Dataset:** Online Retail II — a UK-based online gift-ware retailer, Dec 2009–Dec 2011. This notebook works from an already-cleaned version of the raw dataset (see `cleaning_script.py` in this repo for the cleaning logic and decisions).

**Currency:** all amounts are in pounds (£).

**Tools:** SQL (via DuckDB) for the analysis, including joins, CTEs and window functions; Python/pandas for validation, Tableau Public for the final interactive dashboard.

## Contents
1. [Monthly Revenue Trend](#1.-Monthly-Revenue-Trend)
2. [Top Products: Revenue vs. Quantity](#2.-Top-Products:-Revenue-vs.-Quantity)
3. [Revenue by Country](#3.-Revenue-by-Country)
4. [Average Order Value (AOV)](#4.-Average-Order-Value-(AOV))
5. [Customer Questions in SQL](#5.-Customer-Questions-in-SQL)
6. [Analysis Summary](#Analysis-Summary)


In [1]:
import pandas as pd
import duckdb

In [2]:
data = pd.read_csv("cleaned_retail.csv")
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

con = duckdb.connect()
con.register("retail", data)

print(data.shape)

(804824, 8)


## 1. Monthly Revenue Trend

In [3]:
monthly_revenue = con.sql("""
SELECT
    DATE_TRUNC('month', InvoiceDate) AS Month,
    SUM(Quantity * Price) AS Revenue
FROM retail
GROUP BY Month
ORDER BY Month
""").df()

print(monthly_revenue)

        Month      Revenue
0  2009-12-01   684320.220
1  2010-01-01   538069.372
2  2010-02-01   502669.686
3  2010-03-01   672300.791
4  2010-04-01   591821.772
5  2010-05-01   598287.540
6  2010-06-01   636251.100
7  2010-07-01   586884.030
8  2010-08-01   599898.100
9  2010-09-01   812929.061
10 2010-10-01  1022593.490
11 2010-11-01  1171267.312
12 2010-12-01   884159.470
13 2011-01-01   569335.640
14 2011-02-01   446712.020
15 2011-03-01   590587.850
16 2011-04-01   459076.861
17 2011-05-01   674276.650
18 2011-06-01   660442.580
19 2011-07-01   598543.241
20 2011-08-01   642354.360
21 2011-09-01   949377.141
22 2011-10-01  1017745.160
23 2011-11-01  1158801.060
24 2011-12-01   517757.690


### Data Quality Note: Incomplete Final Month

The dataset's transaction records end on **2011-12-09**, meaning December 2011 contains only 9 days of data compared to a full month (30 days) for every other period in the dataset.

**Impact:** December 2011 shows £517,757 in revenue which is notably lower than November 2011 (£1,158,801) and December 2010 (£884,159). Without this context, this could be misread as a real sales decline.

**Investigation:** Checked the dataset's actual date range (established during data cleaning) and confirmed the December 2011 figure reflects only 9 days of transactions, not a full month.

**Decision:** December 2011 is excluded from month-over-month trend analysis and visualizations to avoid a misleading comparison. This is noted explicitly on the dashboard rather than silently dropping the data point.

In [4]:
monthly_revenue_clean = monthly_revenue[monthly_revenue["Month"] < "2011-12-01"].copy()

monthly_revenue_clean["Revenue_Growth_Pct"] = (
    monthly_revenue_clean["Revenue"].pct_change() * 100
)

print(monthly_revenue_clean)

        Month      Revenue  Revenue_Growth_Pct
0  2009-12-01   684320.220                 NaN
1  2010-01-01   538069.372          -21.371698
2  2010-02-01   502669.686           -6.579019
3  2010-03-01   672300.791           33.746038
4  2010-04-01   591821.772          -11.970686
5  2010-05-01   598287.540            1.092519
6  2010-06-01   636251.100            6.345370
7  2010-07-01   586884.030           -7.759055
8  2010-08-01   599898.100            2.217486
9  2010-09-01   812929.061           35.511191
10 2010-10-01  1022593.490           25.791233
11 2010-11-01  1171267.312           14.538898
12 2010-12-01   884159.470          -24.512580
13 2011-01-01   569335.640          -35.607132
14 2011-02-01   446712.020          -21.538019
15 2011-03-01   590587.850           32.207736
16 2011-04-01   459076.861          -22.267811
17 2011-05-01   674276.650           46.876636
18 2011-06-01   660442.580           -2.051691
19 2011-07-01   598543.241           -9.372403
20 2011-08-01

<br>

**The same calculation in SQL.** `LAG` looks up the revenue of the month before, so the growth can be calculated in one query. The numbers match the pandas version above.

<br>

In [5]:
con.sql("""
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', InvoiceDate) AS Month,
        SUM(Quantity * Price) AS Revenue
    FROM retail
    WHERE InvoiceDate < '2011-12-01'
    GROUP BY Month
)
SELECT
    Month,
    ROUND(Revenue, 2) AS Revenue,
    ROUND((Revenue - LAG(Revenue) OVER (ORDER BY Month)) / LAG(Revenue) OVER (ORDER BY Month) * 100, 1) AS Growth_Pct
FROM monthly
ORDER BY Month
""").df()

,Month,Revenue,Growth_Pct
0,2009-12-01,684320.22,NaN
1,2010-01-01,538069.37,-21.4
2,2010-02-01,502669.69,-6.6
3,2010-03-01,672300.79,33.7
4,2010-04-01,591821.77,-12.0
5,2010-05-01,598287.54,1.1
6,2010-06-01,636251.10,6.3
7,2010-07-01,586884.03,-7.8
8,2010-08-01,599898.10,2.2
9,2010-09-01,812929.06,35.5


### Finding: Consistent Seasonal Pattern

Revenue growth shows a strong, repeating seasonal pattern across both years in the dataset:
- **September**: sharp growth spike (+35.5% in 2010, +47.8% in 2011)
- **October–November**: continued strong growth, peaking ahead of the holidays
- **January**: sharp post-holiday decline (-21.4% in 2010, -35.6% in 2011)

**Why this matters:** the fact that this pattern repeats independently in both 2010 and 2011 — same calendar months, same direction, similar magnitude — rules out a one-time cause (e.g., a single marketing campaign or data anomaly) and strongly supports a genuine, recurring seasonal demand cycle tied to holiday gift shopping. A single year's pattern alone would be an anecdote; two consecutive years showing the same shape is real evidence.

**Business implication:** inventory and marketing planning should anticipate this cycle — building stock ahead of September, and expecting a natural demand drop in January rather than treating it as a concerning decline.

## 2. Top Products: Revenue vs. Quantity

In [6]:
top_products_revenue = con.sql("""
SELECT
    StockCode,
    Description,
    SUM(Quantity * Price) AS Revenue,
    SUM(Quantity) AS Total_Quantity
FROM retail
GROUP BY StockCode, Description
ORDER BY Revenue DESC
LIMIT 10
""").df()

print(top_products_revenue)

  StockCode                         Description    Revenue  Total_Quantity
0     22423            REGENCY CAKESTAND 3 TIER  286486.30         24914.0
1    85123A  WHITE HANGING HEART T-LIGHT HOLDER  252072.46         93640.0
2     23843         PAPER CRAFT , LITTLE BIRDIE  168469.60         80995.0
3    85099B             JUMBO BAG RED RETROSPOT  136980.08         75759.0
4     84879       ASSORTED COLOUR BIRD ORNAMENT  127074.17         79913.0
5      POST                             POSTAGE  126563.04          5333.0
6     47566                       PARTY BUNTING  103880.23         23611.0
7     23166      MEDIUM CERAMIC TOP STORAGE JAR   81416.73         77916.0
8     22086     PAPER CHAIN KIT 50'S CHRISTMAS    79594.33         29477.0
9     79321                       CHILLI LIGHTS   72860.14         15735.0


**Why exclude shipping from this ranking?** Monetary value at the customer level (used in the companion segmentation project) correctly includes postage, since it's real revenue the business collected. But "top products by revenue" is answering a different question — which physical products should the business prioritize stocking, marketing, or negotiating better supplier terms on. Postage isn't a merchandising decision; it's a pass-through fee tied to order volume. Including it here would misleadingly suggest postage is a top-performing product line. **Decision:** exclude non-product codes (`POST`, `DOT`, `C2`) from product-level rankings specifically.

In [7]:
top_products_revenue = con.sql("""
SELECT
    StockCode,
    Description,
    SUM(Quantity * Price) AS Revenue,
    SUM(Quantity) AS Total_Quantity
FROM retail
WHERE StockCode NOT IN ('POST', 'DOT', 'C2')
GROUP BY StockCode, Description
ORDER BY Revenue DESC
LIMIT 10
""").df()

print(top_products_revenue)

  StockCode                         Description    Revenue  Total_Quantity
0     22423            REGENCY CAKESTAND 3 TIER  286486.30         24914.0
1    85123A  WHITE HANGING HEART T-LIGHT HOLDER  252072.46         93640.0
2     23843         PAPER CRAFT , LITTLE BIRDIE  168469.60         80995.0
3    85099B             JUMBO BAG RED RETROSPOT  136980.08         75759.0
4     84879       ASSORTED COLOUR BIRD ORNAMENT  127074.17         79913.0
5     47566                       PARTY BUNTING  103880.23         23611.0
6     23166      MEDIUM CERAMIC TOP STORAGE JAR   81416.73         77916.0
7     22086     PAPER CHAIN KIT 50'S CHRISTMAS    79594.33         29477.0
8     79321                       CHILLI LIGHTS   72860.14         15735.0
9     21137            BLACK RECORD COVER FRAME   67209.44         19629.0


In [8]:
top_products_quantity = con.sql("""
SELECT
    StockCode,
    Description,
    SUM(Quantity) AS Total_Quantity,
    SUM(Quantity * Price) AS Revenue
FROM retail
WHERE StockCode NOT IN ('POST', 'DOT', 'C2')
GROUP BY StockCode, Description
ORDER BY Total_Quantity DESC
LIMIT 10
""").df()

print(top_products_quantity)

  StockCode                         Description  Total_Quantity    Revenue
0     84077   WORLD WAR 2 GLIDERS ASSTD DESIGNS        109169.0   24905.87
1    85123A  WHITE HANGING HEART T-LIGHT HOLDER         93640.0  252072.46
2     23843         PAPER CRAFT , LITTLE BIRDIE         80995.0  168469.60
3     84879       ASSORTED COLOUR BIRD ORNAMENT         79913.0  127074.17
4     23166      MEDIUM CERAMIC TOP STORAGE JAR         77916.0   81416.73
5    85099B             JUMBO BAG RED RETROSPOT         75759.0  136980.08
6     17003                 BROCADE RING PURSE          71129.0   14827.71
7     21977  PACK OF 60 PINK PAISLEY CAKE CASES         55270.0   26733.45
8     84991         60 TEATIME FAIRY CAKE CASES         53495.0   26121.57
9     21212     PACK OF 72 RETROSPOT CAKE CASES         46107.0   22214.26


### Finding: Revenue vs. Volume Tell Different Stories

Comparing "top products by revenue" against "top products by quantity sold" reveals two distinct sets of high-performing products, not one.

**Example:** WORLD WAR 2 GLIDERS (StockCode 84077) sold 109,169 units — the highest quantity of any product — but generated only £24,905.87 in revenue (~£0.23/unit). By contrast, REGENCY CAKESTAND 3 TIER sold far fewer units (24,914) but generated £286,486.30 (~£11.50/unit) — the highest revenue of any single product.

**Business implication:** revenue and volume leaders serve different purposes. High-revenue items (cakestands, bunting) are margin drivers worth prioritizing in marketing and merchandising. High-volume, low-price items (gliders, cake cases) likely drive basket size, impulse purchases, or repeat orders even though each unit contributes little revenue individually. Reporting only one ranking would miss half the picture.

## 3. Revenue by Country

In [9]:
country_revenue = con.sql("""
SELECT
    Country,
    SUM(Quantity * Price) AS Revenue,
    COUNT(DISTINCT "Customer ID") AS Num_Customers
FROM retail
GROUP BY Country
ORDER BY Revenue DESC
""").df()

country_revenue["Pct_of_Total"] = (
    country_revenue["Revenue"] / country_revenue["Revenue"].sum()
) * 100

print(country_revenue.head(15))

            Country       Revenue  Num_Customers  Pct_of_Total
0    United Kingdom  1.464693e+07           5335     83.285267
1              EIRE  6.006616e+05              3      3.415477
2       Netherlands  5.540357e+05             22      3.150353
3           Germany  4.280610e+05            107      2.434037
4            France  3.406877e+05             94      1.937215
5         Australia  1.688347e+05             15      0.960026
6             Spain  1.075398e+05             38      0.611492
7       Switzerland  1.003653e+05             22      0.570697
8            Sweden  8.977004e+04             19      0.510450
9           Denmark  6.986219e+04             12      0.397250
10          Belgium  6.424477e+04             29      0.365308
11         Portugal  5.294364e+04             23      0.301048
12            Japan  4.713839e+04             10      0.268038
13           Norway  4.535298e+04             12      0.257886
14  Channel Islands  4.470376e+04             13      0

### Finding: UK's Revenue Share (83.3%) vs. Transaction Share (90%)

While the UK accounts for 90.0% of transaction rows (and 90.9% of orders), it represents a lower 83.3% of total revenue. This gap is explained by a small number of high-value non-UK accounts — most notably EIRE (Ireland), where just 3 customers generated £600,661 in total revenue, averaging £200,220 per customer versus a dataset-wide median of under £900.

This is consistent with the wholesale/B2B account pattern identified in the companion customer segmentation project — a handful of large business buyers can disproportionately skew country-level revenue figures, the same way they skew individual customer revenue totals. Country-level revenue comparisons should be read with this in mind, rather than assuming they reflect typical per-customer spending.

## 4. Average Order Value (AOV)

In [10]:
data["Revenue"] = data["Quantity"] * data["Price"]

invoice_revenue = (
    data.groupby("Invoice")
    .agg(
        Revenue=("Revenue", "sum"),
        Month=("InvoiceDate", "first"),
        Country=("Country", "first"),
    )
    .reset_index()
)

overall_aov = invoice_revenue["Revenue"].mean()
print("Overall Average Order Value:", round(overall_aov, 2))

Overall Average Order Value: 478.96


In [11]:
aov_by_month = invoice_revenue.groupby(invoice_revenue["Month"].dt.to_period("M"))[
    "Revenue"
].mean()
print(aov_by_month)

Month
2009-12    455.909540
2010-01    558.163249
2010-02    458.222139
2010-03    444.643380
2010-04    449.371125
2010-05    437.344693
2010-06    427.588105
2010-07    429.008794
2010-08    468.304528
2010-09    487.952618
2010-10    483.267245
2010-11    453.979578
2010-12    632.898690
2011-01    578.593130
2011-02    449.408471
2011-03    449.116236
2011-04    402.346066
2011-05    435.860795
2011-06    474.455876
2011-07    451.730748
2011-08    502.624695
2011-09    542.811401
2011-10    532.293494
2011-11    437.448494
2011-12    666.354813
Freq: M, Name: Revenue, dtype: float64


<br>

Before reading the December numbers, check the median order value and the largest single orders. An average can be moved by one very large order.

<br>

In [12]:
con.sql("""
WITH orders AS (
    SELECT
        Invoice,
        DATE_TRUNC('month', MIN(InvoiceDate)) AS Month,
        SUM(Quantity * Price) AS Order_Value
    FROM retail
    GROUP BY Invoice
)
SELECT
    Month,
    COUNT(*) AS Orders,
    ROUND(AVG(Order_Value), 2) AS Average_Order,
    ROUND(MEDIAN(Order_Value), 2) AS Median_Order,
    ROUND(MAX(Order_Value), 2) AS Largest_Order,
    ROUND((SUM(Order_Value) - MAX(Order_Value)) / (COUNT(*) - 1), 2) AS Average_Without_Largest
FROM orders
WHERE Month IN ('2010-11-01', '2010-12-01', '2011-01-01', '2011-11-01', '2011-12-01')
GROUP BY Month
ORDER BY Month
""").df()

,Month,Orders,Average_Order,Median_Order,Largest_Order,Average_Without_Largest
0,2010-11-01,2580,453.98,301.47,18122.17,447.13
1,2010-12-01,1397,632.90,364.04,31770.98,610.59
2,2011-01-01,984,578.59,304.15,77183.60,500.66
3,2011-11-01,2649,437.45,300.32,10584.77,433.62
4,2011-12-01,777,666.35,300.54,168469.60,450.11


### Finding: The December Average Is Real in 2010, but Not in 2011

The average order value looks high in both Decembers: £632.90 in 2010 and £666.35 in 2011, against a typical £430 to £540.

Looking at the median and at the largest orders shows these are two different stories:

- **December 2010 is a real pattern.** The median order also rises, to £364 from about £300 in a normal month. Typical orders were bigger.

- **December 2011 is one order.** A single order of £168,470 (80,995 units of one paper craft item, placed on the last day of the data) pulls the average up. Without it the average is £450, and the median is £301, both normal.

- **January 2011 has the same issue.** One order of £77,184 lifts its average from £501 to £579.

**Lesson:** with a few very large orders in the data, the average can be moved by a single invoice. The median is the safer measure of a typical order.

**Business implication:** there is evidence of larger baskets in December from one year only. That is worth watching, but it is not yet a pattern to plan around.

### AOV by Country

In [13]:
aov_by_country = (
    invoice_revenue.groupby("Country")["Revenue"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)
aov_by_country.columns = ["Avg_Order_Value", "Num_Orders"]
print(aov_by_country.head(15))

             Avg_Order_Value  Num_Orders
Country                                 
Netherlands      2473.373482         224
Australia        1875.940667          90
Lebanon          1693.880000           1
Singapore        1644.770000           8
Denmark          1624.702093          43
Thailand         1535.270000           2
Israel           1488.727143           7
Japan            1428.436061          33
EIRE             1131.189548         531
Switzerland      1115.170444          90
Lithuania        1092.290000           6
Norway           1079.832857          42
Greece           1060.899444          18
RSA               966.870000           2
Sweden            880.098431         102


### Finding: Small Sample Sizes Distort Country-Level AOV

Several countries show high average order values that are based on very few orders — Lebanon (£1,693.88 from just 1 order), Thailand (£1,535.27 from 2 orders), and RSA (£966.87 from 2 orders). These figures are not statistically meaningful and should not be interpreted as representative country-level spending patterns.

Countries with a reasonable order volume (e.g., Netherlands at 224 orders, EIRE at 531 orders, Switzerland at 90 orders) provide more reliable AOV comparisons, though EIRE's figure is still likely influenced by the small number of high-value wholesale accounts identified earlier in this analysis.

**Decision:** for the dashboard, country-level AOV is only displayed for countries with a minimum order threshold (20+ orders), with a note explaining the exclusion, rather than presenting all countries' figures as equally reliable.

In [14]:
aov_by_country_filtered = aov_by_country[aov_by_country["Num_Orders"] >= 20]
print(aov_by_country_filtered)

                 Avg_Order_Value  Num_Orders
Country                                     
Netherlands          2473.373482         224
Australia            1875.940667          90
Denmark              1624.702093          43
Japan                1428.436061          33
EIRE                 1131.189548         531
Switzerland          1115.170444          90
Norway               1079.832857          42
Sweden                880.098431         102
Channel Islands       827.847407          54
Spain                 721.743557         149
Cyprus                684.984444          36
Portugal              615.623721          86
France                562.190875         606
Germany               550.914995         777
Austria               543.093256          43
Finland               532.929091          55
Italy                 507.980469          64
United Kingdom        438.714788       33386
Belgium               434.086284         148
USA                   418.300500          20
Poland    

## 5. Customer Questions in SQL

The first four sections look at totals. This section asks questions about customers and change over time, which need more than a `GROUP BY`: window functions, CTEs (`WITH`) and joins.

<br>

### Question 1: Is the business growing year over year?

`SUM(...) OVER (PARTITION BY Year ...)` gives a running total inside each year. `LAG(Revenue, 12)` looks up the same month one year earlier.

<br>

In [15]:
con.sql("""
WITH monthly AS (
    SELECT
        YEAR(InvoiceDate) AS Year,
        MONTH(InvoiceDate) AS Month,
        SUM(Quantity * Price) AS Revenue
    FROM retail
    GROUP BY Year, Month
)
SELECT
    Year,
    Month,
    ROUND(Revenue) AS Revenue,
    ROUND(SUM(Revenue) OVER (PARTITION BY Year ORDER BY Month)) AS Running_Total,
    ROUND((Revenue - LAG(Revenue, 12) OVER (ORDER BY Year, Month)) / LAG(Revenue, 12) OVER (ORDER BY Year, Month) * 100, 1) AS Change_From_Last_Year_Pct
FROM monthly
WHERE Year >= 2010
ORDER BY Year, Month
""").df()

,Year,Month,Revenue,Running_Total,Change_From_Last_Year_Pct
0,2010,1,538069.0,538069.0,NaN
1,2010,2,502670.0,1040739.0,NaN
2,2010,3,672301.0,1713040.0,NaN
3,2010,4,591822.0,2304862.0,NaN
4,2010,5,598288.0,2903149.0,NaN
5,2010,6,636251.0,3539400.0,NaN
6,2010,7,586884.0,4126284.0,NaN
7,2010,8,599898.0,4726182.0,NaN
8,2010,9,812929.0,5539111.0,NaN
9,2010,10,1022593.0,6561705.0,NaN


<br>

December 2011 is incomplete, so the fair comparison is January to November of each year.

<br>

In [16]:
con.sql("""
SELECT
    YEAR(InvoiceDate) AS Year,
    ROUND(SUM(Quantity * Price)) AS Revenue_Jan_to_Nov
FROM retail
WHERE MONTH(InvoiceDate) <= 11
  AND YEAR(InvoiceDate) >= 2010
GROUP BY Year
ORDER BY Year
""").df()

,Year,Revenue_Jan_to_Nov
0,2010,7732972.0
1,2011,7767253.0


<br>

**Finding:** revenue was flat. January to November 2011 brought in £7,767,253 against £7,732,972 in 2010, a rise of 0.4%. Spring 2011 was weaker than spring 2010 (down 11% to 22% from February to April), and a stronger May to September made up the difference.

**Business implication:** the seasonal peaks are large, but underneath them the business is not growing.

<br>

<br>

### Question 2: Do the best-selling products differ by country?

`RANK() OVER (PARTITION BY Country ...)` ranks products separately inside each country. The five largest countries by revenue are shown.

<br>

In [17]:
con.sql("""
WITH product_sales AS (
    SELECT
        Country,
        StockCode,
        MAX(Description) AS Description,
        SUM(Quantity * Price) AS Revenue
    FROM retail
    WHERE StockCode NOT IN ('POST', 'DOT', 'C2')
    GROUP BY Country, StockCode
),
ranked AS (
    SELECT
        *,
        RANK() OVER (PARTITION BY Country ORDER BY Revenue DESC) AS Rank_In_Country
    FROM product_sales
)
SELECT
    Country,
    Rank_In_Country,
    Description,
    ROUND(Revenue) AS Revenue
FROM ranked
WHERE Rank_In_Country <= 3
  AND Country IN ('United Kingdom', 'EIRE', 'Netherlands', 'Germany', 'France')
ORDER BY Country, Rank_In_Country
""").df()

,Country,Rank_In_Country,Description,Revenue
0,EIRE,1,REGENCY CAKESTAND 3 TIER,15067.0
1,EIRE,2,SET/5 RED SPOTTY LID GLASS BOWLS,7728.0
2,EIRE,3,RETRO SPOT CAKE STAND,6141.0
3,France,1,RABBIT NIGHT LIGHT,7234.0
4,France,2,RED TOADSTOOL LED NIGHT LIGHT,4506.0
5,France,3,ROUND SNACK BOXES SET OF4 WOODLAND,3785.0
6,Germany,1,REGENCY CAKESTAND 3 TIER,13600.0
7,Germany,2,ROUND SNACK BOXES SET OF4 WOODLAND,7920.0
8,Germany,3,ROUND SNACK BOXES SET OF 4 FRUITS,4645.0
9,Netherlands,1,ROUND SNACK BOXES SET OF4 WOODLAND,13315.0


<br>

**Finding:** the Regency cakestand is the top product in the UK, Ireland and Germany. France and the Netherlands are different: their leaders are children's items such as night lights, snack boxes and lunch boxes.

**Business implication:** one product list does not fit every market. Promotions in France and the Netherlands should lead with different items from the UK.

<br>

<br>

### Question 3: How much of the business comes from returning customers?

The first CTE finds each customer's first month. The second lists the months each customer bought in. Joining them shows whether a customer was new or returning in each month.

<br>

In [18]:
con.sql("""
WITH first_order AS (
    SELECT
        "Customer ID" AS Customer_ID,
        DATE_TRUNC('month', MIN(InvoiceDate)) AS First_Month
    FROM retail
    GROUP BY "Customer ID"
),
monthly_customers AS (
    SELECT DISTINCT
        "Customer ID" AS Customer_ID,
        DATE_TRUNC('month', InvoiceDate) AS Order_Month
    FROM retail
)
SELECT
    m.Order_Month,
    COUNT(*) AS Customers,
    SUM(CASE WHEN m.Order_Month = f.First_Month THEN 1 ELSE 0 END) AS New_Customers,
    SUM(CASE WHEN m.Order_Month > f.First_Month THEN 1 ELSE 0 END) AS Returning_Customers,
    ROUND(SUM(CASE WHEN m.Order_Month > f.First_Month THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS Pct_Returning
FROM monthly_customers AS m
JOIN first_order AS f
    ON m.Customer_ID = f.Customer_ID
GROUP BY m.Order_Month
ORDER BY m.Order_Month
""").df()

,Order_Month,Customers,New_Customers,Returning_Customers,Pct_Returning
0,2009-12-01,954,954.0,0.0,0.0
1,2010-01-01,701,368.0,333.0,47.5
2,2010-02-01,771,375.0,396.0,51.4
3,2010-03-01,1053,440.0,613.0,58.2
4,2010-04-01,939,294.0,645.0,68.7
5,2010-05-01,966,255.0,711.0,73.6
6,2010-06-01,1036,267.0,769.0,74.2
7,2010-07-01,925,186.0,739.0,79.9
8,2010-08-01,910,163.0,747.0,82.1
9,2010-09-01,1138,239.0,899.0,79.0


In [19]:
con.sql("""
WITH first_order AS (
    SELECT
        "Customer ID" AS Customer_ID,
        DATE_TRUNC('month', MIN(InvoiceDate)) AS First_Month
    FROM retail
    GROUP BY "Customer ID"
)
SELECT
    YEAR(r.InvoiceDate) AS Year,
    ROUND(SUM(CASE WHEN DATE_TRUNC('month', r.InvoiceDate) > f.First_Month THEN r.Quantity * r.Price ELSE 0 END) * 100.0
          / SUM(r.Quantity * r.Price), 1) AS Pct_Revenue_From_Returning
FROM retail AS r
JOIN first_order AS f
    ON r."Customer ID" = f.Customer_ID
WHERE YEAR(r.InvoiceDate) >= 2010
GROUP BY Year
ORDER BY Year
""").df()

,Year,Pct_Revenue_From_Returning
0,2010,81.7
1,2011,91.1


<br>

**Finding:** returning customers are most of the business. They brought in 81.7% of revenue in 2010 and 91.1% in 2011. New customers per month fell from 250 to 440 in early 2010 to 100 to 220 in 2011.

**A caution:** the data starts in December 2009, so a customer's "first month" is their first month in this data. Some early "new" customers were probably existing customers. That makes the 2010 figures less reliable than the 2011 ones.

**Business implication:** together with Question 1, this explains the flat revenue. Existing customers keep buying, but fewer new ones are arriving.

<br>

<br>

### Question 4: Do new customers come back?

This is a cohort analysis. Customers are grouped by the month of their first order (their cohort). For each cohort, the query counts how many bought again 1, 3, 6 and 12 months later.

The December 2009 cohort is left out, because it mixes real new customers with everyone who was already a customer when the data begins.

<br>

In [20]:
cohorts = con.sql("""
WITH first_order AS (
    SELECT
        "Customer ID" AS Customer_ID,
        DATE_TRUNC('month', MIN(InvoiceDate)) AS First_Month
    FROM retail
    GROUP BY "Customer ID"
),
monthly_customers AS (
    SELECT DISTINCT
        "Customer ID" AS Customer_ID,
        DATE_TRUNC('month', InvoiceDate) AS Order_Month
    FROM retail
),
cohort_activity AS (
    SELECT
        f.First_Month,
        DATE_DIFF('month', f.First_Month, m.Order_Month) AS Months_Later,
        COUNT(DISTINCT m.Customer_ID) AS Customers
    FROM monthly_customers AS m
    JOIN first_order AS f
        ON m.Customer_ID = f.Customer_ID
    GROUP BY f.First_Month, Months_Later
)
SELECT
    First_Month,
    MAX(CASE WHEN Months_Later = 0 THEN Customers END) AS Cohort_Size,
    ROUND(MAX(CASE WHEN Months_Later = 1 THEN Customers END) * 100.0 / MAX(CASE WHEN Months_Later = 0 THEN Customers END), 1) AS Pct_Back_Month_1,
    ROUND(MAX(CASE WHEN Months_Later = 3 THEN Customers END) * 100.0 / MAX(CASE WHEN Months_Later = 0 THEN Customers END), 1) AS Pct_Back_Month_3,
    ROUND(MAX(CASE WHEN Months_Later = 6 THEN Customers END) * 100.0 / MAX(CASE WHEN Months_Later = 0 THEN Customers END), 1) AS Pct_Back_Month_6,
    ROUND(MAX(CASE WHEN Months_Later = 12 THEN Customers END) * 100.0 / MAX(CASE WHEN Months_Later = 0 THEN Customers END), 1) AS Pct_Back_Month_12
FROM cohort_activity
WHERE First_Month >= '2010-01-01'
  AND First_Month <= '2011-11-01'
GROUP BY First_Month
ORDER BY First_Month
""").df()

cohorts

,First_Month,Cohort_Size,Pct_Back_Month_1,Pct_Back_Month_3,Pct_Back_Month_6,Pct_Back_Month_12
0,2010-01-01,368,21.5,31.5,26.9,22.8
1,2010-02-01,375,24.0,29.3,19.2,15.2
2,2010-03-01,440,19.1,24.1,24.8,20.0
3,2010-04-01,294,19.4,16.0,27.6,14.3
4,2010-05-01,255,15.7,17.6,21.2,15.3
5,2010-06-01,267,17.6,20.6,12.7,15.0
6,2010-07-01,186,15.6,29.6,11.3,13.4
7,2010-08-01,163,19.6,32.5,9.8,14.7
8,2010-09-01,239,23.4,13.0,13.8,22.6
9,2010-10-01,375,25.9,12.5,13.1,19.2


In [21]:
# Average across cohorts. A blank means the data ends before that month is reached, and it is left out of the average.
print("Bought again 1 month later: ", round(cohorts["Pct_Back_Month_1"].mean(), 1), "%")
print("Bought again 3 months later:", round(cohorts["Pct_Back_Month_3"].mean(), 1), "%")
print("Bought again 6 months later:", round(cohorts["Pct_Back_Month_6"].mean(), 1), "%")
print("Bought again 12 months later:", round(cohorts["Pct_Back_Month_12"].mean(), 1), "%")

Bought again 1 month later:  20.5 %
Bought again 3 months later: 20.7 %
Bought again 6 months later: 16.8 %
Bought again 12 months later: 16.7 %


<br>

How many new customers ever come back? This counts customers who bought in more than one month. Only customers who joined in 2010 are used, so each has had at least a year to return.

<br>

In [22]:
con.sql("""
WITH customer_months AS (
    SELECT
        "Customer ID" AS Customer_ID,
        DATE_TRUNC('month', MIN(InvoiceDate)) AS First_Month,
        COUNT(DISTINCT DATE_TRUNC('month', InvoiceDate)) AS Months_With_Orders
    FROM retail
    GROUP BY "Customer ID"
)
SELECT
    COUNT(*) AS New_Customers_In_2010,
    SUM(CASE WHEN Months_With_Orders > 1 THEN 1 ELSE 0 END) AS Came_Back,
    ROUND(SUM(CASE WHEN Months_With_Orders > 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS Pct_Came_Back
FROM customer_months
WHERE First_Month >= '2010-01-01'
  AND First_Month <= '2010-12-01'
""").df()

,New_Customers_In_2010,Came_Back,Pct_Came_Back
0,3364,2482.0,73.8


<br>

**Finding:** most new customers do come back, but not quickly. Of the customers who joined in 2010, 73.8% bought again in a later month. Yet only about 1 in 5 buys again the very next month (20.5%), and in any single later month only about 1 in 6 is buying (16.7% a year on). Customers return, but months apart.

**Business implication:** the opportunity is to bring customers back sooner and more often. A follow-up offer in the weeks after an order targets the long gap between purchases.

<br>

<br>

### Question 5: How concentrated is revenue among customers?

`NTILE(10)` splits customers into 10 equal groups by total spend. Group 1 is the top 10% of spenders.

<br>

In [23]:
con.sql("""
WITH customer_totals AS (
    SELECT
        "Customer ID" AS Customer_ID,
        SUM(Quantity * Price) AS Revenue,
        COUNT(DISTINCT Invoice) AS Orders
    FROM retail
    GROUP BY "Customer ID"
),
grouped AS (
    SELECT
        *,
        NTILE(10) OVER (ORDER BY Revenue DESC) AS Spend_Group
    FROM customer_totals
)
SELECT
    Spend_Group,
    COUNT(*) AS Customers,
    ROUND(SUM(Revenue)) AS Revenue,
    ROUND(SUM(Revenue) * 100.0 / SUM(SUM(Revenue)) OVER (), 1) AS Pct_Of_Revenue,
    ROUND(AVG(Orders), 1) AS Average_Orders
FROM grouped
GROUP BY Spend_Group
ORDER BY Spend_Group
""").df()

,Spend_Group,Customers,Revenue,Pct_Of_Revenue,Average_Orders
0,1,586,11226503.0,63.8,26.9
1,2,586,2345190.0,13.3,10.5
2,3,586,1369494.0,7.8,7.2
3,4,586,891594.0,5.1,5.0
4,5,586,619019.0,3.5,3.7
5,6,585,437863.0,2.5,3.0
6,7,585,298627.0,1.7,2.3
7,8,585,204919.0,1.2,1.7
8,9,585,130006.0,0.7,1.4
9,10,585,63248.0,0.4,1.1


<br>

**Finding:** the top 10% of customers (586 people) bring in 63.8% of revenue and place about 27 orders each. The bottom half of customers together bring in 6.5%.

**Business implication:** losing a handful of top customers would hurt more than losing hundreds of small ones. This matches the companion segmentation project, where a "Champions" group of 8% of customers brought in 55% of revenue.

<br>

In [24]:
monthly_revenue_clean.to_csv("monthly_revenue.csv", index=False)
top_products_revenue.to_csv("top_products_revenue.csv", index=False)
top_products_quantity.to_csv("top_products_quantity.csv", index=False)
country_revenue.to_csv("country_revenue.csv", index=False)
aov_by_country_filtered.to_csv("aov_by_country.csv")

print("All tables exported.")

All tables exported.


## Analysis Summary

**1. Revenue by Month (Trend Analysis)**

Computed monthly revenue and month-over-month growth in SQL, cross-checked in pandas. Found a strong, repeating seasonal pattern across both years: sharp growth in September–November (holiday buildup), sharp decline in January (post-holiday slump). Identified December 2011 as an incomplete month (data ends Dec 9) and excluded it from trend analysis to avoid a misleading comparison.

**2. Top Products (Revenue vs. Quantity)**

Compared top 10 products by revenue against top 10 by quantity sold. Found these are largely different products — e.g., WORLD WAR 2 GLIDERS sold the most units (109,169) but generated modest revenue (~£0.23/unit), while REGENCY CAKESTAND 3 TIER generated the most revenue (~£11.50/unit) on far fewer units. Excluded non-product codes (POST, DOT, C2) from this ranking, since shipping isn't a merchandising decision.

**3. Revenue by Country**

UK represents 90% of transaction rows but only 83.3% of revenue. Traced the gap to a small number of high-value non-UK accounts (e.g., EIRE: 3 customers averaging £200,220 each) — consistent with the wholesale/B2B account pattern first identified in the companion segmentation project.

**4. Average Order Value (AOV)**

Calculated overall, by month, and by country. December 2010 shows larger typical orders (the median rises too). December 2011 only looks high because of one £168,470 order, so the median is the safer measure. Several countries' AOV figures were based on too few orders to be meaningful (e.g., Lebanon: £1,693 from a single order), so the country-level AOV table is filtered to a 20+ order minimum.

**5. Customer Questions in SQL**

Used window functions, CTEs and joins to go beyond totals. Revenue was flat year over year (January to November 2011 was 0.4% above 2010). 91% of 2011 revenue came from customers who had bought before. About 74% of new customers come back at some point, but only 1 in 5 does so the following month. The top 10% of customers bring in 64% of revenue.

### Key business themes across all five analyses

Strong, predictable seasonality tied to holiday shopping. A small number of wholesale/B2B accounts distort multiple metrics (customer revenue totals, country revenue, country AOV) — a recurring pattern worth flagging to any stakeholder using this data. Revenue and volume are not interchangeable signals — both are needed for a complete picture of product performance. The business depends on repeat customers who buy months apart, and fewer new customers are arriving, which is why revenue is flat.